In [22]:
#!pip install -r /Users/20s_a02/repositories/EPDE/requirements.txt


In [23]:
import numpy as np
import pandas as pd
import dill as pickle
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import epde

import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances)
import json
from epde import EpdeSearch, EpdeMultisample
import re
import itertools
from pathlib import Path
from typing import List, Dict, Tuple, Any
from data_process import DataProcessor

In [24]:
# ==========================
# === EQUATION DISCOVERER CLASS ===
# ==========================
class EquationDiscoverer:
    """
    Wrapper for EPDE search for discovering system equations.

    Attributes
    ----------
    samples : list
        Input data (normalized coordinates or signals).
    """
    def __init__(self, samples: list):
        self._samples = samples

    # === Discover system equations ===
    def discover_system_equation(self, epde_search_obj: EpdeSearch,
                                 additional_tokens: List, variable_names: list,dimensionality: int = 0) -> Any:
        """Performs EPDE search for system of equations."""
        factors_max_number = {'factors_num': [1, 2], 'probas': [0.85, 0.15]}
        
        trig_tokens = epde.TrigonometricTokens(freq=(0.999, 1.001), dimensionality=dimensionality)
        grid_tokens = epde.GridTokens(['t'], dimensionality=dimensionality)
        
        epde_search_obj.fit(
            data=self._samples,
            variable_names=variable_names,
            max_deriv_order=2,
            equation_terms_max_number=3,
            data_fun_pow=1,
            additional_tokens=additional_tokens+[trig_tokens, grid_tokens],
            equation_factors_max_number=factors_max_number,
            eq_sparsity_interval=(1e-6, 1)
        )

        return epde_search_obj.equations(only_print=False, only_str=False, num=4)


In [25]:
# ==========================
# === EQUATION PROCESSOR CLASS ===
# ==========================
class EquationProcessor:
    """
    Processes EPDE-discovered equations into structured tables for analysis.

    Attributes
    ----------
    regex : compiled regex
        Used to remove frequency patterns from term names.
    """

    def __init__(self):
        self.regex = re.compile(r', freq:\s\d\S\d+')

    # === Helper to update dictionaries ===
    @staticmethod
    def dict_update(d_main: Dict, term: str, coeff: float, k: int) -> Dict:
        """Updates dictionary of terms with new coefficients."""
        str_t = '_r' if '_r' in term else ''
        arr_term = re.sub('_r', '', term).split(' * ')
        perm_set = list(itertools.permutations(range(len(arr_term))))
        structure_added = False

        for p_i in perm_set:
            temp = " * ".join([arr_term[i] for i in p_i]) + str_t
            if temp in d_main:
                if k - len(d_main[temp]) >= 0:
                    d_main[temp] += [0 for _ in range(k - len(d_main[temp]))] + [coeff]
                else:
                    d_main[temp][-1] += coeff
                structure_added = True

        if not structure_added:
            d_main[term] = [0 for _ in range(k)] + [coeff]

        return d_main

   # === Convert equation to table ===
    def equation_table(self, k: int, equation, dict_main: Dict, dict_right: Dict) -> List[Dict]:
        """Creates structured dictionary from EPDE equation."""
        equation_s = equation.structure
        equation_c = equation.weights_final
        text_form_eq = self.regex.sub('', equation.text_form)

        flag = False
        for t_eq in equation_s:
            term = self.regex.sub('', t_eq.name)
            for t in range(len(equation_c)):
                c = equation_c[t]
                if f'{c} * {term} +' in text_form_eq:
                    dict_main = self.dict_update(dict_main, term, c, k)
                    equation_c = np.delete(equation_c, t)
                    break
                elif f'+ {c} =' in text_form_eq:
                    dict_main = self.dict_update(dict_main, "C", c, k)
                    equation_c = np.delete(equation_c, t)
                    break
            if f'= {term}' == text_form_eq[text_form_eq.find('='):] and not flag:
                flag = True
                dict_main = self.dict_update(dict_main, term, -1., k)

        return [dict_main, dict_right]

    # === Convert all objects to table ===
    def object_table(self, res: List, variable_names: List[str],
                     table_main: List[Dict], k: int) -> Tuple[List[Dict], int]:
        """Processes EPDE results into table format."""


        for list_SoEq in res:
            for SoEq in list_SoEq:
                for n, value in enumerate(variable_names):
                    gene = SoEq.vals.chromosome.get(value)
                    table_main[n][value] = self.equation_table(
                        k, gene.value, *table_main[n][value]
                    )
                k += 1
        return table_main, k

    # === Preprocess to DataFrame ===
    def preprocessing_table(self, variable_name: List[str],
                            table_main: List[Dict], k: int) -> pd.DataFrame:
        """Prepares final DataFrame from processed equation tables."""
        data_frame_total = pd.DataFrame()
        for dict_var in table_main:
            for var_name, list_structure in dict_var.items():
                general_dict = {}
                for structure in list_structure:
                    general_dict.update(structure)
                dict_var[var_name] = general_dict

        for dict_var in table_main:
            for var_name, general_dict in dict_var.items():
                for key, value in general_dict.items():
                    if len(value) < k:
                        general_dict[key] = value + [0. for _ in range(k - len(value))]

        data_frame_main = [{i: pd.DataFrame()} for i in variable_name]
        for n, dict_var in enumerate(table_main):
            for var_name, general_dict in dict_var.items():
                data_frame_main[n][var_name] = pd.DataFrame(general_dict)

        for n, var_name in enumerate(variable_name):
            data_frame_temp = data_frame_main[n].get(var_name).copy()
            list_columns = [f'{col}_{var_name}' for col in data_frame_temp.columns]
            data_frame_temp.columns = list_columns
            data_frame_total = pd.concat([data_frame_total, data_frame_temp], axis=1)

        return data_frame_total



In [ ]:
# ==========================
# === ROBOT PROCESSING FUNCTION ===
# ==========================
def process_robot(object_id, normalized_coord_data,pwm_signal, lev = 'micro', force = 'without', num_parts=1, 
                  output_dir=Path("EPDE_output"), save_flag:bool = True):
    """Processes robot data, splits into parts, discovers equations, and saves results."""
    print(f"\n Processing robot {object_id} with {num_parts} parts...")
    print(len(list(normalized_coord_data.values())[0]))
    cut_number = min(200, len(list(normalized_coord_data.values())[0]))
    cut_points = np.zeros((cut_number, len(object_id), 2))

    # === Directories ===
    if lev == 'micro':
        robot_dir = output_dir / f"robot_{object_id[0]}/{num_parts}_parts/{force}_force"
        robot_dir.mkdir(parents=True, exist_ok=True)

    if lev == 'meso':
        ids = '_'.join([str(i) for i in object_id])
        robot_dir = output_dir / f"robot_{ids}/{num_parts}_parts/{force}_force"
        robot_dir.mkdir(parents=True, exist_ok=True)

    for i, robot_id in enumerate(object_id):
        coords = np.array(normalized_coord_data[robot_id][:cut_number])
        cut_points[:, i, :] = coords
    

    # === Split data into parts ===
    part_size = cut_number // num_parts
    boundaries = [i * part_size for i in range(num_parts)] + [cut_number]

    for part_idx in range(num_parts):
        start_idx = boundaries[part_idx]
        end_idx = boundaries[part_idx + 1]
        part_points = cut_points[start_idx:end_idx,:,:]
        # pwm_part = np.array(pwm_signal[start_idx:end_idx])
        samples = []
        for i in range(len(object_id)):

            x = part_points[:, i, 0]
            y = part_points[:, i, 1]

            samples.append(x)
            samples.append(y)

        variable_names = []
        for i in range(len(object_id)):

            variable_names.append(f"x{i+1}")
            variable_names.append(f"y{i+1}")

        t_data = np.arange(len(part_points))
        
        print(f"\n--- Optimizing part {part_idx + 1}/{num_parts} ({start_idx}:{end_idx}) ---")

        # === Objective function for Optuna ===
        def objective(trial):
            poly_window = trial.suggest_int("poly_window", 5, 11, step=2)
            population_size = trial.suggest_int("population_size", 4, 14, step=2)
            sigma = trial.suggest_int("sigma", 0, 5, step=1)
            min_boundary = poly_window // 2 + 1
            max_boundary = 7
            boundary = trial.suggest_int("boundary", min_boundary, max_boundary)

            epde_search_obj_system = EpdeSearch(
            use_solver=False,
            boundary=boundary,
            coordinate_tensors=[t_data]
            )
            epde_search_obj_system.set_moeadd_params(population_size=population_size, training_epochs=5)
            epde_search_obj_system.set_preprocessor(
                default_preprocessor_type='poly',
                preprocessor_kwargs={'use_smoothing': True, 'sigma': sigma,'polynomial_window': poly_window}
            )
            
            '''
            external_force_token = CacheStoredTokens(
                token_type='ext_force',
                token_labels=['PWM'],
                token_tensors={'PWM': pwm_part},
                params_ranges={'power': (1, 1)},
                params_equality_ranges=None,
                meaningful=True
            )
            ''' 
            
            equation_system = EquationDiscoverer(samples)
            part_equations = equation_system.discover_system_equation(
                epde_search_obj_system,
                additional_tokens=[],
                variable_names = variable_names,
                dimensionality=0
            )

            values_per_level = [[solution.obj_fun for solution in level] for level in part_equations]
            flat_obj = [vals for level in values_per_level for vals in level]
            for level_idx, level in enumerate(values_per_level):
                print(f"Level {level_idx}:")
                for sol_idx, obj_values in enumerate(level):
                    print(f"Solution {sol_idx}: {obj_values}")

            best_score = np.inf
            n = len(variable_names)

            for obj_values in flat_obj:
                score = sum(obj_values[:n])
                constraint_sum = sum(obj_values[n:2*n])
                if constraint_sum > 0.1 * n:
                    score += 100 
                best_score = min(best_score, score)

            print(f"✅ Trial {trial} finish")
            return best_score

        # === Optuna study ===
        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=10)
        best_params = study.best_params
        best_score = study.best_value
        print(f"✅ Part {part_idx + 1}: Best params {best_params}, score={best_score}")
        
        fig1 = plot_optimization_history(study)
        fig1.show()
        if save_flag:
            history_plot_path = robot_dir /f"part_{part_idx + 1}_system_history_plot.html"
            fig1.write_html(history_plot_path)
        fig2 = plot_param_importances(study)
        fig2.show()
        if save_flag:
            importances_plots_path = robot_dir /f"part_{part_idx + 1}_system_importances_plots.html"
            fig2.write_html(importances_plots_path)

        # === Save plots and params ===
        params_path = robot_dir /f"part_{part_idx + 1}_system_best_params.json"
        with open(params_path, "w") as f:
            json.dump({"best_params": best_params, "score": best_score}, f, indent=4)

        # === Final EPDE system with best params ===
        best_boundary = best_params["boundary"]
        best_window = best_params["poly_window"]
        best_sigma = best_params["sigma"]
        best_population_size = best_params["population_size"]

        epde_search_obj_system = EpdeSearch(
            use_solver=False,
            boundary=best_boundary,
            coordinate_tensors=[t_data]
        )
        epde_search_obj_system.set_moeadd_params(population_size=best_population_size, training_epochs=10)
        epde_search_obj_system.set_preprocessor(
            default_preprocessor_type='poly',
            preprocessor_kwargs={'use_smoothing': True, 'sigma': best_sigma,'polynomial_window': best_window}
        )
        equation_system = EquationDiscoverer(samples)
        part_equations = equation_system.discover_system_equation(
            epde_search_obj_system,
            additional_tokens=[],
            variable_names = variable_names,
            dimensionality=0
        )

        '''
        external_force_token = CacheStoredTokens(
            token_type='ext_force',
            token_labels=['PWM'],
            token_tensors={'PWM': pwm_part},
            params_ranges={'power': (1, 1)},
            params_equality_ranges=None,
            meaningful=True
        )
        pwm_derivs_token = ExternalDerivativesTokens(
            'PWM_derivs',                     # имя токена
            boundary=4,                       # как и у других данных
            time_axis=0,                      # ось времени (0, т.к. одномерная зависимость)
            base_token_label='PWM',           # имя базового токена
            token_tensor=pwm_signal,          # сам сигнал
            max_orders=2,                     # до какой производной считать (1 -> dPWM/dt, 2 -> d²PWM/dt²)
            deriv_method='poly',              # метод вычисления производных
            deriv_method_kwargs={
                'smooth': True,
                'grid': [t_data]
            },
            params_ranges={'power': (1, 1)},  # оставляем без возведения в степень
            params_equality_ranges=None,
            meaningful=True
        )
        '''
        
        values_per_level = [
        [solution.obj_fun for solution in level]
                            for level in part_equations]
        for level_idx, level in enumerate(values_per_level):
            print(f"Level {level_idx}:")
            for sol_idx, obj_values in enumerate(level):
                print(f"Solution {sol_idx}: {obj_values}")
        for level_idx, level in enumerate(part_equations):
            print(f"Level {level_idx}:")
            for sol_idx, solution in enumerate(level):
                print(f"  Solution {sol_idx}:")
                for eq in solution.vals:
                    print("    ", eq.text_form)
                for eq in solution.vals:
                    print("    ", eq.latex_form)
        
        if save_flag:
            output_file = robot_dir /"obj_func_and_equations.txt"

            with open(output_file, "w") as f:
                # Сначала выводим значения obj_fun
                values_per_level = [
                    [solution.obj_fun for solution in level]
                    for level in part_equations
                ]
                for level_idx, level in enumerate(values_per_level):
                    f.write(f"Level {level_idx}:\n")
                    for sol_idx, obj_values in enumerate(level):
                        f.write(f"Solution {sol_idx}: {obj_values}\n")
                
                # Теперь выводим сами уравнения
                for level_idx, level in enumerate(part_equations):
                    f.write(f"\nLevel {level_idx}:\n")
                    for sol_idx, solution in enumerate(level):
                        f.write(f"  Solution {sol_idx}:\n")
                        # Если у тебя solution.vals существует (EPDE объект)
                        if hasattr(solution, "vals"):
                            for eq in solution.vals:
                                f.write(f"    {eq.text_form}\n")
                            for eq in solution.vals:
                                f.write(f"    {eq.latex_form}\n")
                        else:
                            # Если solution это просто список строк или numpy array
                            for eq in solution:
                                f.write(f"    {eq}\n")

            print(f"Output saved to {output_file}")

        # === Process equations into table ===
        equation_processor = EquationProcessor()
        variable_names = variable_names
        table_main = [{i: [{}, {}]} for i in variable_names]
        k = 0
        vals_flat = []
        for elem in part_equations:
            vals_flat.extend(elem)

        table_main, k = equation_processor.object_table(
            [vals_flat], variable_names, table_main, k
        )
        print(f"\nSystem of DE {part_idx + 1} (startpoint {start_idx}-{end_idx - 1}):")
        for i, eq in enumerate(part_equations):
            print(f"EQ {i + 1}:")
            print(eq)

        # === Save table ===
        if save_flag:
            frame_main = equation_processor.preprocessing_table(variable_names, table_main, k)
            output_path = robot_dir /f"system_{part_idx + 1}.csv"
            frame_main.to_csv(output_path, sep=',', encoding='utf-8')
            print(f"Saved: {output_path}")
    
# ==========================
# === MAIN FUNCTION ===
# ==========================
def main(raw_data, target_robots:List = None, robots_form: str = 'circle' , num_parts:int =1, lev:str  = 'micro', 
         particle_flag:bool =False, save_flag:bool = True):
    processor = DataProcessor(raw_data)
    processor.extract_data()
    pwm_signal=None
    #pwm_signal = np.array(processor.angle_data[target_robot])
    normalized_coord_data = processor.normalize_all_coordinates()
    robots_to_process = []

    if (particle_flag) and (robots_form == 'circle'):
        for target_robot in target_robots:
            particle_data = dict(np.load(f"{Path(robots_form)}/particle_coordinates.npz"))
            particle_coords = np.array(particle_data[f'{target_robot}'][:200])
            x_particle, y_particle = particle_coords[:, 0], particle_coords[:, 1]
            normalized_coord_data, key = processor.transform_new_robot(x_particle, y_particle, target_robot+100)
            robots_to_process += [key]
    else:
        robots_to_process = target_robots    
    

    if lev == 'micro':
        output_dir = Path(robots_form)/"EPDE_output_micro"

    if lev == 'meso':
        output_dir = Path(robots_form)/"EPDE_output_mezo"
        
    process_robot(robots_to_process, normalized_coord_data, pwm_signal,num_parts=num_parts,lev = lev, output_dir=output_dir,
                  save_flag = save_flag)

robots_form = 'oval' #'oval' or 'circle'
if robots_form == 'oval':
    with open('oval/data/oval_data_[30_bots_PWM_1_exp_1].pickle', 'rb') as file: 
        raw_data = pickle.load(file)
elif robots_form == 'circle':
    with open('circle/data/circle_data_00_330_[30_bots_PWM_10_15cw_15ccw_D_41cm].MP4.pickle', 'rb') as file: 
        raw_data = pickle.load(file)

level = 'micro' #'micro', 'meso_cluster', 'meso_union'
if level == 'micro':
    lev = 'micro'
elif level in ['meso_cluster', 'meso_union']:
    lev = 'meso'

path_levels_robots_ids = f'{Path(robots_form)}/levels_robots_ids.json'
with open(path_levels_robots_ids, "r") as f:
    levels_robots_ids = json.load(f)

particle_flag = False

save_flag = False

for target_robots in levels_robots_ids[level]:
    main(raw_data, target_robots=target_robots,robots_form=robots_form, num_parts=1, lev = lev, 
         particle_flag=particle_flag, save_flag = save_flag)
# trials, traning_epoch


[I 2026-04-13 22:34:30,868] A new study created in memory with name: no-name-95df3959-ced0-4905-8a0f-e360a53cd835



 Processing robot [2, 5, 7, 12, 13, 20, 25, 26, 28, 32, 33, 39, 41, 50, 51, 55, 57, 60, 62, 65, 66, 68, 69, 71, 72, 73, 75, 76, 77, 80] with 1 parts...
921

--- Optimizing part 1/1 (0:200) ---
setting builder with <epde.optimizers.builder.StrategyBuilder object at 0x300eaad50>
setting builder with <epde.optimizers.builder.StrategyBuilder object at 0x300eaad50>
trig_token_params: VALUES = (0, 0)
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (188,) derivs_tensor.shape (188, 2)
Size of linked labels is 3
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (188,) derivs_tensor.shape (188, 2)
Size of linked labels is 6
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (188,) derivs_tensor.shape (188, 2)
Size of linked labels is 9
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (188,) derivs_tensor.shape (188, 2)
Size of linked labels is 12
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (188,) derivs_tensor.shape (188, 2)
Si

[I 2026-04-13 23:18:09,821] Trial 0 finished with value: 156.92509548559445 and parameters: {'poly_window': 7, 'population_size': 10, 'sigma': 4, 'boundary': 6}. Best is trial 0 with value: 156.92509548559445.


[1.24639550e+00 1.00041834e+00 8.88265045e-01 1.07977715e+00
 6.34585211e-01 9.95873059e-01 9.50347743e-01 1.00911012e+00
 1.01023686e+00 9.81907145e-01 1.11824427e+00 9.92869431e-01
 9.75193907e-01 3.43220387e-01 3.06761427e-01 1.03726033e+00
 8.91061313e-01 9.57111219e-01 1.05142954e+00 1.11373240e+00
 7.34496345e-01 1.29952167e+00 1.19053643e+00 1.06829996e+00
 1.01493247e+00 5.23282987e-01 1.06878691e+00 1.13947030e+00
 5.13792638e-01 9.32900597e-01 1.04480248e+00 9.84014242e-01
 9.89561209e-01 1.19103138e+00 1.40523005e+00 4.69682089e-01
 9.99478683e-01 9.96554132e-01 1.04406397e+00 1.30021861e+00
 3.61117299e+00 1.14640115e+00 1.02489135e+00 9.92080617e-01
 7.69906617e-01 9.18998749e-01 1.12348684e+00 7.76286661e-01
 1.09480821e+00 9.58759742e-01 7.15836586e-01 9.91325560e-01
 1.22364080e+00 1.10952830e+00 1.01171718e+00 1.16178674e+00
 9.39957059e-01 1.21736622e+00 1.10216290e+00 1.13061178e+00
 6.24117350e+01 9.03200223e+00 1.25250260e+00 3.99428770e-01
 7.22736733e-01 1.034353

[I 2026-04-14 01:06:35,372] Trial 1 finished with value: 159.81877662214137 and parameters: {'poly_window': 5, 'population_size': 14, 'sigma': 5, 'boundary': 6}. Best is trial 0 with value: 156.92509548559445.


...

[I 2026-04-14 05:27:30,321] Trial 8 finished with value: 159.3241365511258 and parameters: {'poly_window': 7, 'population_size': 8, 'sigma': 1, 'boundary': 7}. Best is trial 7 with value: 155.216026842328.


[1.04865536e+00 1.06011041e+00 1.00260927e+00 1.13758813e+00
 9.88757751e-01 3.12637126e+00 9.88755544e-01 9.77854373e-01
 1.09769477e+00 9.75515284e-01 1.05715455e+00 1.01093854e+00
 6.78539046e-01 1.06194010e+00 4.74545347e-01 9.87593853e-01
 1.00441211e+00 8.59656213e-01 1.41711986e+00 9.81574175e-01
 9.71063306e-01 9.97759462e-01 1.07469996e+00 9.83739128e-01
 1.03603271e+00 1.02725540e+00 3.26194612e+00 1.05266422e+00
 3.63641472e-01 6.74224608e-01 1.03855530e+00 1.01187564e+00
 8.15668308e-01 1.00768570e+00 6.06016649e-01 2.05024256e+00
 9.64974596e-01 9.92484275e-01 1.01676724e+00 9.92930004e-01
 1.23875885e+00 9.74961242e-01 9.18845641e-01 9.57213393e-01
 9.81421832e-01 1.07056468e+00 7.97912208e-01 6.43743327e-01
 8.94926058e-01 9.99001711e-01 7.26005706e-01 9.90326573e-01
 9.65827691e-01 9.76263011e-01 1.01449052e+00 1.03246590e+00
 9.41576332e-01 2.29271151e+00 1.04955177e+00 1.22350002e+00
 9.22192197e-01 1.81884911e+00 6.43982857e+00 8.19822639e+01
 2.39566205e+00 1.134137

[I 2026-04-14 06:11:53,952] Trial 9 finished with value: 160.67454440540408 and parameters: {'poly_window': 5, 'population_size': 10, 'sigma': 2, 'boundary': 7}. Best is trial 7 with value: 155.216026842328.


[5.03098269e-01 1.00615479e+00 8.30924012e-01 9.99643190e-01
 6.86325821e-01 9.85520981e-01 1.07346367e+00 8.20147010e-01
 1.03102479e+00 7.38235682e-01 1.02931232e+00 9.77811801e-01
 1.03728360e+00 4.81767119e-01 1.24970371e+00 9.26034534e-01
 7.31279376e-01 9.80449992e-01 1.04136101e+00 9.85572050e-01
 1.08260420e+00 9.23098370e-01 9.45810608e-01 1.03791600e+00
 1.01526610e+00 9.75206319e-01 1.04176080e+00 8.87335685e-01
 1.11442290e+00 1.38291783e+00 9.54310707e-01 9.99794534e-01
 1.00048516e+00 9.92032145e-01 8.12409247e-01 6.60027928e-01
 1.02663544e+00 9.64592052e-01 1.00263962e+00 9.90588615e-01
 1.10896935e+00 1.07115589e+00 9.97145248e-01 1.00536651e+00
 1.00043794e+00 9.61388321e-01 5.19635272e-01 1.10773265e+00
 9.45803375e-01 9.82994484e-01 5.24497648e-01 1.00172984e+00
 8.08699620e-01 1.08750541e+00 9.68507143e-01 1.01282790e+00
 1.00912825e+00 1.15506526e+00 1.06504864e+00 1.13554632e+00
 1.98846936e+01 1.02641512e+01 1.40814288e+00 2.41075273e+01
 5.63096623e-01 3.877888

setting builder with <epde.optimizers.builder.StrategyBuilder object at 0x17ee81b10>
setting builder with <epde.optimizers.builder.StrategyBuilder object at 0x17ee81b10>
trig_token_params: VALUES = (0, 0)
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (186,) derivs_tensor.shape (186, 2)
Size of linked labels is 3
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (186,) derivs_tensor.shape (186, 2)
Size of linked labels is 6
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (186,) derivs_tensor.shape (186, 2)
Size of linked labels is 9
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (186,) derivs_tensor.shape (186, 2)
Size of linked labels is 12
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (186,) derivs_tensor.shape (186, 2)
Size of linked labels is 15
Deriv orders after definition [[0], [0, 0]]
200
initial_shape (186,) derivs_tensor.shape (186, 2)
Size of linked labels is 18
Deriv orders after definition [[0], [0, 0]

In [ ]:
# ==========================
# === COLLECT SCORES FUNCTION ===
# ==========================
def collect_scores(robot_id=79, root="./output"):
    """Collects all best scores for a robot across parts and forces."""
    root = Path(root)
    scores = []

    for parts_dir in (root / f"robot_{robot_id}").glob("*_parts"):
        n_parts = int(parts_dir.name.split("_")[0])

        for force_dir in parts_dir.glob("*_force"):
            part_scores = []
            for part in range(1, n_parts + 1):
                json_path = force_dir / f"part_{part}_system_best_params.json"
                if not json_path.exists():
                    continue
                with open(json_path, "r") as f:
                    data = json.load(f)
                    score = data.get("score")
                    if score is not None:
                        part_scores.append(score)
            if part_scores:
                avg_score = np.mean(part_scores)
                scores.append({
                    "robot_id": robot_id,
                    "parts_dir": parts_dir.name,
                    "force_dir": force_dir.name,
                    "score": avg_score,
                    "raw_scores": part_scores
                })
    return scores

scores = collect_scores(robot_id=79, root="./output")
scores_sorted = sorted(scores, key=lambda x: (x["parts_dir"], x["force_dir"]))
for s in scores_sorted:
    print(f"Robot {s['robot_id']}, {s['parts_dir']}/{s['force_dir']}: "
          f"score = {s['score']:.5f}, raw scores = {s['raw_scores']}")


Robot 79, 2_parts/without_force: score = 0.25797, raw scores = [0.3576760073279146, 0.15825808675294764]
Robot 79, 4_parts/without_force: score = 0.39153, raw scores = [0.7992378348143865, 0.45043637597957065, 0.16182902947885283, 0.15462449962119434]
Robot 79, 3_parts/without_force: score = 1.72626, raw scores = [2.85775422944278, 0.3562654219715613, 1.964772397721285]
Robot 79, 1_parts/without_force: score = 1.41417, raw scores = [1.4141659411632166]
